In [ ]:
import pandas as pd
import numpy as np
import re
import math
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('coffee_data.csv')
df['agtron_whole'] = df['agtron'].str.extract(r'^(\d+)').astype(float)
print(f'Shape: {df.shape}')

# --- extract process_method from coffee name ---
PROCESS_KEYWORDS = [
    'Carbonic Maceration', 'Anaerobic', 'Semi-Washed', 'Pulped Natural',
    'Wet-Hulled', 'Natural', 'Washed', 'Honey', 'Experimental',
]

VARIETY_KEYWORDS = [
    'Pink Bourbon', 'Yellow Bourbon', 'Red Bourbon',
    'SL28', 'SL34', 'Geisha', 'Gesha', 'Bourbon', 'Typica',
    'Pacamara', 'Caturra', 'Catuai', 'Sidra', 'Maragogipe',
    'Wush Wush', 'Mocca', 'Laurina', 'Maracaturra',
    'Mundo Novo', 'Peaberry', 'Marsellesa', 'Heirloom', 'Mejorado',
]

COUNTRY_KEYWORDS = [
    'Ethiopia', 'Kenya', 'Colombia', 'Panama', 'Guatemala', 'Costa Rica',
    'Honduras', 'El Salvador', 'Nicaragua', 'Peru', 'Brazil', 'Yemen',
    'Jamaica', "Hawai'i", 'Hawaii', 'Indonesia', 'Papua New Guinea',
    'Rwanda', 'Burundi', 'Uganda', 'Tanzania', 'Malawi', 'Zambia',
    'Ecuador', 'Bolivia', 'Mexico', 'China', 'Taiwan', 'India',
    'Myanmar', 'Thailand', 'Vietnam',
]

def extract_process(name):
    if pd.isna(name): return 'Unknown'
    for kw in PROCESS_KEYWORDS:
        if re.search(rf'\b{re.escape(kw)}\b', str(name), re.IGNORECASE):
            return kw
    return 'Unknown'

def extract_variety(name):
    if pd.isna(name): return 'Unknown'
    for kw in VARIETY_KEYWORDS:
        if re.search(rf'\b{re.escape(kw)}\b', str(name), re.IGNORECASE):
            return kw
    return 'Unknown'

def extract_country(origin):
    if pd.isna(origin): return 'Unknown'
    s = str(origin).lower()
    for country in COUNTRY_KEYWORDS:
        if country.lower() in s:
            return country
    return 'Unknown'

def extract_region(origin):
    if pd.isna(origin): return 'Unknown'
    parts = str(origin).split(',')
    region = parts[0].strip()
    return region if len(region) > 3 else 'Unknown'

df['process_method'] = df['name'].apply(extract_process)
df['variety']        = df['name'].apply(extract_variety)

origin_col = 'coffee_origin' if 'coffee_origin' in df.columns else None
df['coffee_country'] = df[origin_col].apply(extract_country) if origin_col else 'Unknown'
df['coffee_region']  = df[origin_col].apply(extract_region)  if origin_col else 'Unknown'

print('process_method:', df['process_method'].value_counts().head(5).to_dict())
print('variety:',        df['variety'].value_counts().head(5).to_dict())
print('coffee_country:', df['coffee_country'].value_counts().head(5).to_dict())

In [ ]:
NUMERIC_FEATURES = ['agtron_whole', 'aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score']
CAT_FEATURES     = ['coffee_country', 'coffee_region', 'process_method', 'variety']
FEATURES         = ['roast_level'] + NUMERIC_FEATURES + CAT_FEATURES

roast_order = [['Light', 'Medium-Light', 'Medium', 'Medium-Dark', 'Dark']]

preprocessor = ColumnTransformer(transformers=[
    ('roast', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(categories=roast_order, handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scale',  StandardScaler()),
    ]), ['roast_level']),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('categorical', Pipeline([
        ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=False)),
    ]), CAT_FEATURES),
])

print('Preprocessor defined.')

In [ ]:
# Fit preprocessor on full dataset
X_raw = preprocessor.fit_transform(df[FEATURES])

# Build weight vector matching ColumnTransformer output order:
#   [roast(1)] + [numeric(7)] + [OHE columns per categorical feature]
NUMERIC_WEIGHTS = [0.8, 1.5, 1.2, 1.0, 2.0, 2.0, 1.0]  # agtron, aroma, structure, body, flavour, aftertaste, score
CAT_WEIGHTS     = [1.0, 0.8, 1.5, 1.5]                   # country, region, process, variety

# Use the ColumnTransformer's full output names to count OHE columns per feature.
# Names are prefixed 'categorical__' and may use either the column name or 'x{i}' format.
ct_names  = preprocessor.get_feature_names_out()
cat_names = [c for c in ct_names if c.startswith('categorical__')]

n_cols_per_cat = []
for i, feat in enumerate(CAT_FEATURES):
    n = sum(
        1 for c in cat_names
        if c.startswith(f'categorical__{feat}_') or c.startswith(f'categorical__x{i}_')
    )
    n_cols_per_cat.append(n)

ohe_weights = []
for n, w in zip(n_cols_per_cat, CAT_WEIGHTS):
    ohe_weights.extend([w] * n)

weight_vector = np.array([0.8] + NUMERIC_WEIGHTS + ohe_weights)

# Weighted feature matrix stored for similarity queries
X = X_raw * weight_vector

print(f'Feature matrix  : {X.shape}')
print(f'Weight vector   : {weight_vector.shape}')
print(f'OHE cols/feature: {dict(zip(CAT_FEATURES, n_cols_per_cat))}')

In [ ]:
def recommend_coffee(
        roast_level, aroma, structure, body, flavour, aftertaste,
        score=95, agtron_whole=None,
        coffee_country=None, coffee_region=None,
        process_method=None, variety=None,
        top_n=5):
    user_input = pd.DataFrame([{
        'roast_level':    roast_level,
        'agtron_whole':   agtron_whole,
        'aroma':          aroma,
        'structure':      structure,
        'body':           body,
        'flavour':        flavour,
        'aftertaste':     aftertaste,
        'score':          score,
        'coffee_country': coffee_country  or 'Unknown',
        'coffee_region':  coffee_region   or 'Unknown',
        'process_method': process_method  or 'Unknown',
        'variety':        variety          or 'Unknown',
    }])

    query       = preprocessor.transform(user_input) * weight_vector
    similarities = cosine_similarity(query, X)[0]
    top_indices = similarities.argsort()[::-1][:top_n]

    names = df['name'].values
    print(f'Top {top_n} recommended coffees:')
    for rank, idx in enumerate(top_indices, 1):
        print(f'  {rank}. {names[idx]:<55s} {similarities[idx]:.2f}')

print('recommend_coffee() ready.')

In [ ]:
# Held-out evaluation — preprocessor fitted on full data, query test against train
df_tr, df_te = train_test_split(df, test_size=0.2, random_state=42)

X_tr     = preprocessor.transform(df_tr[FEATURES]) * weight_vector
X_te     = preprocessor.transform(df_te[FEATURES]) * weight_vector
names_tr = df_tr['name'].values
names_te = df_te['name'].values

top5 = top10 = 0
mrr_total = ndcg_total = 0.0

for i in range(len(X_te)):
    query     = X_te[i:i+1]
    true_name = names_te[i]
    sims      = cosine_similarity(query, X_tr)[0]
    ranked    = names_tr[np.argsort(sims)[::-1]]

    if true_name in ranked:
        rank = int(np.where(ranked == true_name)[0][0]) + 1
        if rank <= 5:  top5  += 1
        if rank <= 10: top10 += 1
        mrr_total  += 1.0 / rank
        if rank <= 10:
            ndcg_total += 1.0 / math.log2(rank + 1)

n = len(names_te)
print(f'Top-5  hit rate : {top5  / n:.4f}')
print(f'Top-10 hit rate : {top10 / n:.4f}')
print(f'MRR             : {mrr_total / n:.4f}')
print(f'NDCG@10         : {ndcg_total / n:.4f}')

In [ ]:
recommend_coffee('Light', aroma=9, structure=9, body=9, flavour=10, aftertaste=9,
                 process_method='Natural', variety='Geisha')

In [9]:
import joblib

joblib.dump(preprocessor, 'preprocessor.joblib')
joblib.dump(X, 'feature_matrix.joblib')
joblib.dump(weight_vector, 'weight_vector.joblib')
df[['name']].to_csv('coffee_names.csv', index=True)

print('Saved: preprocessor, feature_matrix, weight_vector, coffee_names')

Saved: preprocessor, feature_matrix, weight_vector, coffee_names


## V5 — Weighted Cosine Retrieval Recommender

V5 reframes the model as a content-based coffee recommender. Instead of predicting one exact coffee bean, it builds weighted feature vectors from tasting scores, roast data, score, origin, process and variety, then retrieves the most similar coffees using cosine nearest neighbours. This better matches the real problem because many coffees share near-identical tasting profiles, making exact classification unreliable.

### Feature weights

| Feature | Weight | Source |
|:---|:---:|:---|
| flavour | 2.0 | Tasting score |
| aftertaste | 2.0 | Tasting score |
| aroma | 1.5 | Tasting score |
| process_method | 1.5 | Extracted from coffee name |
| variety | 1.5 | Extracted from coffee name |
| structure | 1.2 | Tasting score |
| body | 1.0 | Tasting score |
| score | 1.0 | Overall review score |
| coffee_country | 1.0 | Extracted from coffee_origin |
| agtron_whole | 0.8 | Roast measurement |
| roast_level | 0.8 | Roast category |
| coffee_region | 0.8 | Extracted from coffee_origin |

### V5 Sample Output

Query: `roast=Light, aroma=9, structure=9, body=9, flavour=10, aftertaste=9, process_method=Natural, variety=Geisha`

```
Top 5 recommended coffees:
  1. Ka'ū Geisha Champagne Natural                           0.94
  2. Panama Don Pachi Natural Geisha                         0.93
  3. Panama Lino Esmeralda Geisha Natural                    0.93
  4. Panama Finca Kalithea Natural Geisha                    0.91
  5. Panama Esmeralda Geisha Natural                         0.91
```

**Notes:**
- Recommendations are thematically coherent — Natural Geisha query surfaces Natural Geisha coffees
- Meaningful score spread (0.94 → 0.91) vs V4's collapse of positions 5–20 all at 0.91
- Top 4 results identical to KNN V5; #5 differs by one bean — expected, both models use cosine similarity on the same weighted vectors
- Process and variety weights (1.5×) are driving the thematic clustering

### Changes from V4
- Dropped `roaster` and `roaster_location` (high-cardinality noise)
- Added `process_method`, `variety`, `coffee_country`, `coffee_region` extracted from text
- Feature weights applied after preprocessing — flavour/aftertaste/aroma/process/variety drive similarity
- OHE uses `min_frequency=10` to collapse rare one-off categories
- Missing categoricals filled with `'Unknown'` (not most-frequent imputation)
- Evaluation uses Top-5/10 hit rate, MRR, NDCG@10 — not exact-match accuracy

## V4 Sample Output

Query: `roast=Light, aroma=9, structure=9, body=9, flavour=10, aftertaste=9`

```
Top 20 recommended coffees:
  1. Kona S12 Kaffa Washed                                   0.95
  2. Colombia Cauca El Paraiso                               0.94
  3. Yemen Lot 106                                           0.92
  4. Panama Elida Natural Lot #13                            0.91
  5. Kona Orange                                             0.91
  6. Yemen Microlot                                          0.91
  7. Kenya Thageini                                          0.91
  8. David Mburu Kenya                                       0.91
  9. Wilton Benitez Java                                     0.91
  10. Hawaii Kilauea Volcano Nano-Lot                         0.91
  11. Wilton Benitez Geisha                                   0.91
  12. Colombia Triple-Anaerobic Honey El Triunfo Pink Bourbon 0.91
  13. Colombia Villa Betulia CM Sidra                         0.91
  14. Yemen Haraaz                                            0.91
  15. Dodora Double                                           0.91
  16. Guatemala Washed El Injerto EI05 Mocca                  0.91
  17. Guatemala Santa Felisa Wild Yeast Natural Gesha         0.91
  18. Ethiopia Natural Sidama Blue Donkey Lot 2               0.91
  19. Ethiopia Sidama Bensa Hamasho Tadisa Washed Slow Dry G1 0.91
  20. Kenya Machakos AA Top Washed                            0.91
```

**Notes:**
- Top 4 show meaningful differentiation (0.95 → 0.91); positions 5–20 all tie at 0.91
- Tie collapse is the same issue as in KNN: discrete integer tasting scores cause many coffees to map to near-identical vectors
- V5 addresses this with feature weights and richer categorical features (process, variety, country)